# KDD Process Volcano Data Analysis

**Dataset :**  Volcano Events

**Team Members:**
- Hannah-May LITTIERE
- Timothée JOLIOT
- Axel JUILLARD
- Sandrine DANIEL
- Léo-Paul CHAUVIGNE

## Imports

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from prophet import Prophet
import plotly.graph_objects as go 
import plotly.express as px
import plotly.io as pio
from dash import Dash, dcc, html, Input, Output

/home/leopa/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Data Collection and Exploration

In this section, we implement the first step of the KDD process: Data Collection and Preprocessing. The dataset used is the "Significant Volcanic Eruption Database" provided by the National Center for Environmental Information (NCEI). It contains the data for over 600 eruptions from 4360 BC to the present. Each eruption is detailed with attributes such as location, time, VEI (Volcanic Explosivity Index), and impact metrics (deaths, damage).

**Objectives of this step:**
1.  Load the data into a Pandas DataFrame.
2.  Clean the data by handling missing values and correct data types.
3.  Summarize the data (shape, types, missing values) to understand its quality.
4.  Establish rankings between variables to identify initial patterns.

In [2]:
#Display settings to ensure we see all columns during exploration
pd.set_option('display.max_columns', None)

In [3]:
def load_raw_data(filepath):
    """
    Reads the TSV file.
    Input: filepath (str)
    Output: raw dataframe (pd.DataFrame)
    """
    try:
        df = pd.read_csv(filepath, sep='\t')
        print("File loaded.")
        return df
    except FileNotFoundError:
        print("File not found. Please check the path.")
        return None


# Place the volcano-events.tsv file in the same directory as this notebook
file_path = "volcano-events.tsv"
df_raw = load_raw_data(file_path)

# We display the first 5 rows of the dataframe for a first peak at the data
df_raw.head(5)

File loaded.


,Search Parameters,Year,Mo,Dy,Tsu,Eq,Name,Location,Country,Latitude,Longitude,Elevation (m),Type,VEI,Agent,Deaths,Death Description,Missing,Missing Description,Injuries,Injuries Description,Damage ($Mil),Damage Description,Houses Destroyed,Houses Destroyed Description,Total Deaths,Total Death Description,Total Missing,Total Missing Description,Total Injuries,Total Injuries Description,Total Damage ($Mil),Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description
0,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,-4360.0,NaN,NaN,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,-4350.0,NaN,NaN,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,P,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
3,NaN,-4050.0,NaN,NaN,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-4000.0,NaN,NaN,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,T,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN


With this first look at the data we notice a few things:
- The first row is filled with NaNs and the search parameter value is '[]'. It seems this row is an artefact, we will remove it.
- There appears to be a lot of NaNs in the data, we will need to take this into account.
- The date data is split between 3 columns, 'Year', 'Mo' and 'Dy'. We will need to take this into account.
- In addition when looking at the Year column we also see that there are negative years, we will therefore need to add a way to display Years with B.C. and A.D. for readability in the dashboard.
- The agent column here has a value 'P' and a value 'T', these are characteristic and each letter stands for something, we will therefore have to understand what each letter stands for.

In [4]:
df_raw['Agent'].unique()

array([nan, 'P', 'T', 'P,T', 'W', 'M', 'T,L', 'S,W', 'F', 'I', 'T,F',
       'P,T,M', 'A', 'L', 'E', 'M,T', 'S', 'P,E,M', 'G,W,T', 'G,T',
       'T,P,W,I', 'T,P', 'm', 'G', 'W,S', 'W,T', 'P,I', 'T,W,I', 'F,P,I',
       'W,A', 'M,L,T,G', 'T,W', 'T, m', 'P,M', 'P,M,E', 'P,W,A,I', 'G,I',
       'W,M', 'T,A', 'L,G', 'W,P,I', 'A,P', 'I,M', '?', 'W,P', 'P,W',
       'T,G', 'P,m', 'I,S,A,T', 'S,L', 'P,W,T', 'A,F', 'P,T,W,I', 'E,I',
       'T,L,I', 'P,T,M,A', 'M,m,P', 'P,M,I', 'P,T,G', 'M,P', 'T,m', 'T,M',
       'P,L', 'P,M,T', 'A,T,W', 'P,T,M,S', 'I,T', 'L,T', 'P,m,I', 'T,I',
       'A,W', 'T,G,M', 'S,T,G', 'A,P,T', 'P,T,W', 'G,W, S', 'L,I,S',
       'M,G,T', 'T,P,L'], dtype=object)

Looking in further detail at the agent column we can see that sometimes an eruption has multiple agents. Here the agents are separated by a comma. We will need to seperate them into binary columns to better analyse them.

After further research we can see that the agents are as follows:
Based on the NOAA Significant Volcanic Eruption Database and the classification standards by Simkin and Siebert (1994), the Agent column indicates the specific volcanic hazard or phenomenon that caused fatalities, injuries, or damage during an eruption. 

- A : Avalanche (Debris)
- E : Electrical (Lightning)
- F : Floods
- G : Gas
- I : Indirect (Starvation, Disease)
- L : Lava Flow
- M : Mudflow (Lahar) (Primary) (upper case indicates the event happened at the same time as the eruption)
- m : Mudflow (Lahar) (Secondary) (lower case indicates the event happened after the eruption)
- P : Pyroclastic Flow
- S : Seismic
- T : Tephra/Ash fall (falling volcanic rock and ash).
- W : Waves


Now we notice something interesting. Tsunamis (Waves) and Earthquakes (Seismic) are listed in agents aswell as there standalone columns although this is discutable because the names are not exactly the same. We therefore need to check if the agents and the standalone columns' values overlap before continuing.

In [5]:
def divide_agents(df):
    """
    Splits the comma-separated 'Agent' column into individual binary columns for easier analysis.
    intput: df (pd.DataFrame)
    output: df_agent_divided (pd.DataFrame)
    """

    # Convert to string, handle NaNs as empty strings, and remove spaces (e.g. "T, m" -> "T,m")
    agents_cleaned = df['Agent'].fillna('').astype(str).str.replace(' ', '')
    
    # We Split and Create the Binary Columns
    # 'sep=,' tells pandas to treat comma as the delimiter
    agent_dummies = agents_cleaned.str.get_dummies(sep=',')
    
    # Rename columns for clarity (e.g. "P" -> "Agent_P")
    agent_dummies = agent_dummies.add_prefix('Agent_')
    
    # Concatenate the new columns to the original dataframe
    df_agent_divided = pd.concat([df, agent_dummies], axis=1)

    df_agent_divided = df_agent_divided.drop(columns=['Agent'])
    
    return df_agent_divided

df_agent_divided = divide_agents(df_raw)
df_agent_divided.dtypes

Search Parameters                      object
Year                                  float64
Mo                                    float64
Dy                                    float64
Tsu                                   float64
Eq                                    float64
Name                                   object
Location                               object
Country                                object
Latitude                              float64
Longitude                             float64
Elevation (m)                         float64
Type                                   object
VEI                                   float64
Deaths                                float64
Death Description                     float64
Missing                               float64
Missing Description                   float64
Injuries                              float64
Injuries Description                  float64
Damage ($Mil)                         float64
Damage Description                

In [6]:
def check_tsu_eq_consistency(df_agent_divided):
    """
    Specialized EDA function to check whether tsunami and earthquake data overlaps with the agent column. We also include a check to see if floods overlap with tsunamis as tsunamis could also be considered as causing floods.
    Input: df_agent_divided (pd.DataFrame) with agent columns divided
    Output: None
    """
    
    # We count the number of non-null values in the 'Tsu' and 'Eq' columns
    tsu_col_count = df_agent_divided['Tsu'].notna().sum()
    eq_col_count = df_agent_divided['Eq'].notna().sum()

    # We count the number of 'T', 'S', 'W', and 'F' codes in the 'Agent' column
    agent_S_col_count = df_agent_divided['Agent_S'].sum()
    agent_W_col_count = df_agent_divided['Agent_W'].sum()
    agent_F_col_count = df_agent_divided['Agent_F'].sum() # Added Flood count

    # Print the counts
    print(f"\nCOUNTS:")
    print(f"Total non-null 'Tsu' column values: {tsu_col_count}")
    print(f"Total 'W' (Waves) codes in Agent:   {agent_W_col_count}")
    print(f"Total 'F' (Floods) codes in Agent:  {agent_F_col_count}\n") # Added Flood count print

    print(f"Total non-null 'Eq' column values:  {eq_col_count}")
    print(f"Total 'S' (Seismic) codes in Agent: {agent_S_col_count}\n")

    

    # Now we check if the columns match
    print(f"\nCONSISTENCY CHECK:")
    
    # Check Tsunami (Agent_W vs Tsu) 
    # Mismatch A: Has 'Tsu' Data (Not Null) BUT 'Agent_W' is 0
    mismatch_tsu_A = df_agent_divided[df_agent_divided['Tsu'].notna() & (df_agent_divided['Agent_W'] == 0)]
    
    # Mismatch B: Has 'Agent_W' (== 1) BUT 'Tsu' is Missing (NaN)
    mismatch_tsu_B = df_agent_divided[(df_agent_divided['Agent_W'] == 1) & df_agent_divided['Tsu'].isna()]

    if mismatch_tsu_A.empty and mismatch_tsu_B.empty:
        print("Tsunami Data (Agent_W): The data is consistent.")
        print(" Every row with a Tsunami flag (W) has a corresponding 'Tsu' data and vice versa.")
    else:
        print("Tsunami Data (Agent_W): The data is not consistent.")
        if not mismatch_tsu_A.empty:
            print(f"   - {len(mismatch_tsu_A)} rows have 'Tsu' data but are missing the 'W' agent.")
        if not mismatch_tsu_B.empty:
            print(f"   - {len(mismatch_tsu_B)} rows have 'W' agent but are missing 'Tsu' data.")

    # Check Flood (Agent_F vs Tsu) - Added consistency check for floods and tsunamis
    # Mismatch A: Has 'Tsu' Data (Not Null) BUT 'Agent_F' is 0
    mismatch_flood_tsu_A = df_agent_divided[df_agent_divided['Tsu'].notna() & (df_agent_divided['Agent_F'] == 0)]
    
    # Mismatch B: Has 'Agent_F' (== 1) BUT 'Tsu' is Missing (NaN)
    mismatch_flood_tsu_B = df_agent_divided[(df_agent_divided['Agent_F'] == 1) & df_agent_divided['Tsu'].isna()]

    print(f"\n")
    if mismatch_flood_tsu_A.empty and mismatch_flood_tsu_B.empty:
        print("Flood Data (Agent_F) vs Tsunami Data (Tsu): The data is consistent.")
        print(" Every row with a Flood flag (F) has corresponding 'Tsu' data and vice versa.")
    else:
        print("Flood Data (Agent_F) vs Tsunami Data (Tsu): The data is not consistent.")
        if not mismatch_flood_tsu_A.empty:
            print(f"   - {len(mismatch_flood_tsu_A)} rows have 'Tsu' data but are missing the 'F' agent.")
        if not mismatch_flood_tsu_B.empty:
            print(f"   - {len(mismatch_flood_tsu_B)} rows have 'F' agent but are missing 'Tsu' data.")


    # Check Earthquake (Agent_S vs Eq) 

    # Mismatch A: Has 'Eq' Data (Not Null) BUT 'Agent_S' is 0
    mismatch_eq_A = df_agent_divided[df_agent_divided['Eq'].notna() & (df_agent_divided['Agent_S'] == 0)]
    
    # Mismatch B: Has 'Agent_S' (== 1) BUT 'Eq' is Missing (NaN)
    mismatch_eq_B = df_agent_divided[(df_agent_divided['Agent_S'] == 1) & df_agent_divided['Eq'].isna()]

    print(f"\n")
    if mismatch_eq_A.empty and mismatch_eq_B.empty:
        print("Earthquake Data: The data is consistent.")
        print(" Every row with an Earthquake flag has a corresponding 'S' agent and vice versa.")
    else:
        print("Earthquake Data: The data is not consistent.")
        if not mismatch_eq_A.empty:
            print(f"   - {len(mismatch_eq_A)} rows have 'Eq' data but are missing the 'S' agent.")
        if not mismatch_eq_B.empty:
            print(f"   - {len(mismatch_eq_B)} rows have 'S' agent but are missing 'Eq' data.")

# We run the function on df_agent_divided
check_tsu_eq_consistency(df_agent_divided)


COUNTS:
Total non-null 'Tsu' column values: 182
Total 'W' (Waves) codes in Agent:   64
Total 'F' (Floods) codes in Agent:  11

Total non-null 'Eq' column values:  79
Total 'S' (Seismic) codes in Agent: 25


CONSISTENCY CHECK:
Tsunami Data (Agent_W): The data is not consistent.
   - 119 rows have 'Tsu' data but are missing the 'W' agent.
   - 1 rows have 'W' agent but are missing 'Tsu' data.


Flood Data (Agent_F) vs Tsunami Data (Tsu): The data is not consistent.
   - 182 rows have 'Tsu' data but are missing the 'F' agent.
   - 11 rows have 'F' agent but are missing 'Tsu' data.


Earthquake Data: The data is not consistent.
   - 62 rows have 'Eq' data but are missing the 'S' agent.
   - 8 rows have 'S' agent but are missing 'Eq' data.


This result is again interesting, almost all Tsunamis are associated with with waves and almost all earthquakes are associated with seismic activity even though there are a few outliers. Whatmore is that all floods happen without tsunamis and vice versa therefor tsunamis and floods are not associated with each other in this dataset.

Allthough tsunamis and waves overlapp and earthquakes and seismic activity overlap, we will keep them separate to better understand the data.

In [7]:
def show_structure(df):
    """
    Prints the shape and data types of the dataset.
    Input: df (pd.DataFrame)
    """ 
    rows, cols = df.shape
    print(f"Number of Rows: {rows}")
    print(f"Number of Columns: {cols}")
 
    print("Data Types:")
    print(df.dtypes)

show_structure(df_agent_divided)

Number of Rows: 888
Number of Columns: 47
Data Types:
Search Parameters                      object
Year                                  float64
Mo                                    float64
Dy                                    float64
Tsu                                   float64
Eq                                    float64
Name                                   object
Location                               object
Country                                object
Latitude                              float64
Longitude                             float64
Elevation (m)                         float64
Type                                   object
VEI                                   float64
Deaths                                float64
Death Description                     float64
Missing                               float64
Missing Description                   float64
Injuries                              float64
Injuries Description                  float64
Damage ($Mil)             

Looking at the dtypes of the data we notice a few more things:

- The 'Tsu' and 'Eq' columns which stand for Tsunami and Earthquake are of type float64. They probably are binary columns, so they could be converted to boolean or just treated as such. We will still need to check the data to be sure.
- The date columns 'Year', 'Mo', and 'Dy' are of type float64. This is counter-intuitive for date components (which should be integers) and confirms the presence of missing values (NaN). In pandas, a column of integers containing even a single NaN is automatically cast to float.
- The columns ending in "Description" (e.g., Death Description, Damage Description, Houses Destroyed Description) are also float64. This is a significant finding: it implies these columns contain numeric codes (likely ordinal scales like 1, 2, 3, 4) rather than actual text descriptions. If they contained text, their dtype would be object. They also might have NaNs, which we will need to check.
- The impact columns like Deaths, Injuries, and Houses Destroyed are float64 as well. This reinforces that we must distinguish between 0 (confirmed no casualties) and NaN (unknown data). Treating NaN as 0 without verification could dangerously skew the analysis (e.g., assuming an ancient eruption had 0 deaths just because records are missing).
- Total Damage is in millions of US dollars, we will need to be careful to not forget that.

In [8]:
def check_description_values(df):
    """
    Checks the unique values of all columns ending with 'Description'.
    Input: df (pd.DataFrame)
    Output: None
    """
    print("Inspecting Unique Values in Description Columns")
    
    # Filter columns that end with 'Description'
    description_cols = [col for col in df.columns if col.endswith('Description')]
    
    if not description_cols:
        print("No columns ending with 'Description' found.")
        return

    for col in description_cols:
        # Get unique values, dropping NaNs to see the codes clearly
        unique_vals = sorted(df[col].dropna().unique())
        print(f"Unique values in '{col}': {unique_vals}")

# Run the function
check_description_values(df_agent_divided)

Inspecting Unique Values in Description Columns
Unique values in 'Death Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Missing Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Injuries Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Damage Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Houses Destroyed Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Total Death Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Total Missing Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Total Injuries Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Total Damage Description': [np.float64

Looking at the description columns we see that they represent ordinal scales like we thought. This can be redundant with the columns that represent the actual number of fatalities, injuries, missing persons... But maybe they are interesting since we may lack precise data for some of the older eruptions.



In [9]:
def analyze_information_loss(df):
    """
    Analyzes potential information loss for all Description columns.
    It counts rows where the Quantitative value is Missing (NaN) 
    but the Qualitative Description exists (Not NaN).
    
    Input: df (pd.DataFrame)
    Output: Prints a summary table
    """
    
    # 1. Define the pairs: (Quantitative Column, Description Column)
    pairs = [
        ('Deaths', 'Death Description'),
        ('Missing', 'Missing Description'),
        ('Injuries', 'Injuries Description'),
        ('Damage ($Mil)', 'Damage Description'),
        ('Houses Destroyed', 'Houses Destroyed Description'),
        ('Total Deaths', 'Total Death Description'),
        ('Total Missing', 'Total Missing Description'),
        ('Total Injuries', 'Total Injuries Description'),
        ('Total Damage ($Mil)', 'Total Damage Description'),
        ('Total Houses Destroyed', 'Total Houses Destroyed Description')
    ]
    
    print(f"{'Quantitative Column':<25} | {'Missing Qty':<12} | {'Description Col':<30} | {'Hidden Data (Loss)'}")
    print("-" * 90)
    
    for qty_col, desc_col in pairs:
        # Verify columns exist in dataframe to avoid errors
        if qty_col not in df.columns or desc_col not in df.columns:
            print(f"{qty_col:<25} | {'N/A':<12} | {desc_col:<30} | Column Not Found")
            continue
            
        # Count total missing quantitative values
        missing_qty = df[qty_col].isna().sum()
        
        # Count "Hidden Data": Rows where Qty is NaN AND Desc is NOT NaN
        # These are the events you lose if you simply drop the description column.
        hidden_data = df[df[qty_col].isna() & df[desc_col].notna()]
        loss_count = len(hidden_data)
        
        print(f"{qty_col:<25} | {missing_qty:<12} | {desc_col:<30} | {loss_count}")

analyze_information_loss(df_agent_divided)


Quantitative Column       | Missing Qty  | Description Col                | Hidden Data (Loss)
------------------------------------------------------------------------------------------
Deaths                    | 445          | Death Description              | 127
Missing                   | 876          | Missing Description            | 4
Injuries                  | 787          | Injuries Description           | 24
Damage ($Mil)             | 864          | Damage Description             | 224
Houses Destroyed          | 838          | Houses Destroyed Description   | 78
Total Deaths              | 423          | Total Death Description        | 131
Total Missing             | 876          | Total Missing Description      | 4
Total Injuries            | 784          | Total Injuries Description     | 30
Total Damage ($Mil)       | 860          | Total Damage Description       | 241
Total Houses Destroyed    | 829          | Total Houses Destroyed Description | 96


Looking at this result we can therefore see that dropping the description columns is not a good idea. We therefore have to find a way to understand this ordinal orders. To do this we will visualise the distribution of the description columns compared to the numerical values they represent in rows where we have both.

In [ ]:

def visualize_distributions_by_category(df):
    """
    Creates box plots to visualize the distribution of various quantitative metrics
    for each corresponding description category.
    This helps justify why using the Median is better than the Mean (due to outliers).
    """
    metrics_to_visualize = [
        ('Deaths', 'Death Description'),
        ('Missing', 'Missing Description'),
        ('Injuries', 'Injuries Description'),
        ('Damage ($Mil)', 'Damage Description'),
        ('Houses Destroyed', 'Houses Destroyed Description'),
        ('Total Deaths', 'Total Death Description'),
        ('Total Missing', 'Total Missing Description'),
        ('Total Injuries', 'Total Injuries Description'),
        ('Total Damage ($Mil)', 'Total Damage Description'),
        ('Total Houses Destroyed', 'Total Houses Destroyed Description')
    ]

    for quantitative_col, description_col in metrics_to_visualize:
        df_filtered = df.dropna(subset=[quantitative_col, description_col])

        if df_filtered.empty:
            print(f"No data available for {quantitative_col} and {description_col}. Skipping plot.")
            continue
            
        fig = px.box(df_filtered, 
                     x=description_col, 
                     y=quantitative_col, 
                     points="all", 
                     title=f"Distribution of {quantitative_col} by Category (Log Scale)",
                     log_y=True,
                     labels={description_col: 'Category Description', quantitative_col: f'{quantitative_col} (Log Scale)'},
                     color=description_col
                    )
        
        fig.update_layout(showlegend=False)
        fig.show()

visualize_distributions_by_category(df_agent_divided)

In [52]:
def analyze_all_description_pairs(df):
    """
    Loops through multiple metric/description pairs to verify if they all follow 
    the ordinal scale pattern (1-4).
    """
    # We want to investigate the pairs as followed :  (Metric Column, Description Column)
    pairs_to_check = [
        ('Deaths', 'Death Description'),
        ('Missing', 'Missing Description'),
        ('Injuries', 'Injuries Description'),
        ('Damage ($Mil)', 'Damage Description'),
        ('Houses Destroyed', 'Houses Destroyed Description'),
        ('Total Deaths', 'Total Death Description'),
        ('Total Missing', 'Total Missing Description'),
        ('Total Injuries', 'Total Injuries Description'),
        ('Total Damage ($Mil)', 'Total Damage Description'),
        ('Total Houses Destroyed', 'Total Houses Destroyed Description')
    ]

    for metric_col, desc_col in pairs_to_check:
        df_filtered = df.dropna(subset=[metric_col, desc_col])
        # Check if columns exist in the dataframe before proceeding
        if metric_col in df_filtered.columns and desc_col in df_filtered.columns:
            print(f"Analysis: {metric_col}  vs.  {desc_col}")

            
            #Statistical Check
            # Group by the description (1, 2, 3, 4) and see the range of the actual numbers
            stats = df.groupby(desc_col)[metric_col].agg(
                Count='count', 
                Min='min', 
                Median='median', 
                Max='max',
                Mean='mean'
            )
            print(stats)
            
            # Filter data for plotting
            plot_df = df.dropna(subset=[metric_col, desc_col]).copy()
            plot_df['Category'] = plot_df[desc_col].astype(str) # Convert to string for categorical plotting
            
            fig = px.box(df_filtered, 
                        x=desc_col, 
                        y=metric_col, 
                        points="all", 
                        title=f"Distribution of {metric_col} by Category (Log Scale)",
                        log_y=True,
                        labels={desc_col: 'Category Description', metric_col: f'{metric_col} (Log Scale)'},
                        color=desc_col
                        )
            fig.show()
            
        else:
            print(f"Skipping pair {metric_col}/{desc_col}: Columns not found.")


analyze_all_description_pairs(df_agent_divided)

Analysis: Deaths  vs.  Death Description
                   Count     Min  Median      Max         Mean
Death Description                                             
1.0                  336     1.0     2.0     51.0     7.217262
2.0                   26    54.0    68.0    120.0    72.538462
3.0                   43   106.0   215.0   1000.0   311.395349
4.0                   38  1001.0  2000.0  30000.0  4650.342105


Analysis: Missing  vs.  Missing Description
                     Count     Min  Median     Max    Mean
Missing Description                                       
1.0                      5     2.0     9.0    44.0    15.6
2.0                      1    78.0    78.0    78.0    78.0
3.0                      2   120.0   174.5   229.0   174.5
4.0                      2  1500.0  1627.5  1755.0  1627.5


Analysis: Injuries  vs.  Injuries Description
                      Count     Min  Median      Max         Mean
Injuries Description                                             
1.0                      80     1.0     5.0     50.0    11.075000
2.0                       7    53.0    65.0     86.0    64.857143
3.0                      11   152.0   277.0   1000.0   352.545455
4.0                       2  1972.0  5986.0  10000.0  5986.000000


Analysis: Damage ($Mil)  vs.  Damage Description
                    Count    Min  Median       Max        Mean
Damage Description                                            
1.0                     2   0.04    0.52     1.000    0.520000
2.0                     6   2.00    2.75     3.564    2.810667
3.0                     7  10.00   15.00    20.000   15.102857
4.0                     9  67.00  325.00  2000.000  510.222222


Analysis: Houses Destroyed  vs.  Houses Destroyed Description
                              Count     Min  Median     Max         Mean
Houses Destroyed Description                                            
1.0                              19     1.0    10.0    75.0    18.894737
2.0                               5    60.0    63.0    90.0    71.000000
3.0                              14   144.0   300.0   800.0   377.428571
4.0                              12  1109.0  3085.0  9000.0  3593.166667


Analysis: Total Deaths  vs.  Total Death Description
                         Count     Min  Median      Max         Mean
Total Death Description                                             
1.0                        342     1.0     2.5    103.0     7.558480
2.0                         27    54.0    68.0    120.0    71.555556
3.0                         52   106.0   220.5   1000.0   331.423077
4.0                         44  1001.0  2000.0  60000.0  7097.090909


Analysis: Total Missing  vs.  Total Missing Description
                           Count     Min  Median     Max         Mean
Total Missing Description                                            
1.0                            6     2.0     9.5    44.0    14.666667
2.0                            1    78.0    78.0    78.0    78.000000
3.0                            2   120.0   174.5   229.0   174.500000
4.0                            2  1500.0  1627.5  1755.0  1627.500000


Analysis: Total Injuries  vs.  Total Injuries Description
                            Count     Min   Median      Max          Mean
Total Injuries Description                                               
1.0                            80     1.0      6.5     50.0     11.837500
2.0                             8    53.0     64.5     86.0     64.750000
3.0                            13   112.0    250.0   1000.0    345.923077
4.0                             3  1972.0  10000.0  31943.0  14638.333333


Analysis: Total Damage ($Mil)  vs.  Total Damage Description
                          Count   Min   Median     Max        Mean
Total Damage Description                                          
1.0                           1   1.0    1.000     1.0    1.000000
2.0                           8   2.0    2.750     4.0    2.908000
3.0                           7   5.0   15.000    20.0   14.714286
4.0                          12  67.0  192.684  2000.0  424.364000


Analysis: Total Houses Destroyed  vs.  Total Houses Destroyed Description
                                    Count     Min  Median     Max         Mean
Total Houses Destroyed Description                                            
1.0                                    20     1.0    12.5    75.0    18.700000
2.0                                     6    60.0    69.0    90.0    71.666667
3.0                                    19   144.0   300.0   871.0   388.210526
4.0                                    14  1109.0  3085.0  9000.0  3734.428571


After analysing the distributions and to get the most out of the data, we will input the median values of the description categories into rows where the numerical values are missing. We chose the median because it is less affected by outliers than other alternatives. This way we will get a better overview of the data especially for older eruptions.

In [12]:
def impute_missing_numericals(df):
    """
    Imputes missing values for ALL quantitative columns (Deaths, Injuries, Damage, etc.)
    based on their corresponding Description columns using the Median strategy.
    
    For each category (1, 2, 3, 4) in a Description column, we calculate the 
    median of the non-missing values in the corresponding Quantitative column, 
    and use that to fill the missing values.
    input: df (pandas.DataFrame)
    output: df_imputed (pandas.DataFrame)
    """

    # Create a copy
    df_imputed = df.copy()
    
    # List of (Quantitative Column, Description Column) pairs
    pairs = [
        ('Deaths', 'Death Description'),
        ('Missing', 'Missing Description'),
        ('Injuries', 'Injuries Description'),
        ('Damage ($Mil)', 'Damage Description'),
        ('Houses Destroyed', 'Houses Destroyed Description'),
        ('Total Deaths', 'Total Death Description'),
        ('Total Missing', 'Total Missing Description'),
        ('Total Injuries', 'Total Injuries Description'),
        ('Total Damage ($Mil)', 'Total Damage Description'),
        ('Total Houses Destroyed', 'Total Houses Destroyed Description')
    ]
    
    print("--- Imputation Report ---")
    
    for qty_col, desc_col in pairs:
        # Check if columns exist in the dataframe to avoid errors
        if qty_col not in df_imputed.columns or desc_col not in df_imputed.columns:
            continue
            
        # 1. Calculate Medians for each description category dynamically
        # This creates a Series like {1.0: 2.0, 2.0: 68.0, ...} specific to this column
        # We assume ordinal scales 1, 2, 3, 4
        median_map = df_imputed.groupby(desc_col)[qty_col].median()
        
        # 2. Identify rows to fill: Qty is NaN AND Desc is NOT NaN
        mask = df_imputed[qty_col].isna() & df_imputed[desc_col].notna()
        
        count_to_fill = mask.sum()
        
        if count_to_fill > 0:
            # 3. Apply Imputation
            # Map the description value to the calculated median
            # Note: If a category has no median (e.g., category 4 exists but has NO numbers), it remains NaN
            df_imputed.loc[mask, qty_col] = df_imputed.loc[mask, desc_col].map(median_map)
            
            print(f"[{qty_col}] Filled {count_to_fill} missing values using calculated medians: {median_map.to_dict()}")
        else:
            print(f"[{qty_col}] No imputable missing values found.")

    return df_imputed

df_imputed = impute_missing_numericals(df_agent_divided)

--- Imputation Report ---
[Deaths] Filled 127 missing values using calculated medians: {1.0: 2.0, 2.0: 68.0, 3.0: 215.0, 4.0: 2000.0}
[Missing] Filled 4 missing values using calculated medians: {1.0: 9.0, 2.0: 78.0, 3.0: 174.5, 4.0: 1627.5}
[Injuries] Filled 24 missing values using calculated medians: {1.0: 5.0, 2.0: 65.0, 3.0: 277.0, 4.0: 5986.0}
[Damage ($Mil)] Filled 224 missing values using calculated medians: {1.0: 0.52, 2.0: 2.75, 3.0: 15.0, 4.0: 325.0}
[Houses Destroyed] Filled 78 missing values using calculated medians: {1.0: 10.0, 2.0: 63.0, 3.0: 300.0, 4.0: 3085.0}
[Total Deaths] Filled 131 missing values using calculated medians: {1.0: 2.5, 2.0: 68.0, 3.0: 220.5, 4.0: 2000.0}
[Total Missing] Filled 4 missing values using calculated medians: {1.0: 9.5, 2.0: 78.0, 3.0: 174.5, 4.0: 1627.5}
[Total Injuries] Filled 30 missing values using calculated medians: {1.0: 6.5, 2.0: 64.5, 3.0: 250.0, 4.0: 10000.0}
[Total Damage ($Mil)] Filled 241 missing values using calculated medians: {

In [13]:
def show_missing_data(df):
    """
    Calculates and prints the number of missing values per column.
    Input: df (pd.DataFrame)
    """
    # Requirement: Show the number of missing data points in each column 
    missing_values = df.isnull().sum()
    
    # Filter to show only columns that actually have missing data
    missing_only = missing_values[missing_values > 0]
    
    print("Missing Data Points per Column:")
    if not missing_only.empty:
        print(missing_only)
    else:
        print("No missing values found in the dataset columns.")

show_missing_data(df_agent_divided)

Missing Data Points per Column:
Search Parameters                     887
Year                                    1
Mo                                    132
Dy                                    192
Tsu                                   706
Eq                                    809
Name                                    1
Location                                1
Country                                 1
Latitude                                1
Longitude                               1
Elevation (m)                           1
Type                                    1
VEI                                   183
Deaths                                445
Death Description                     318
Missing                               876
Missing Description                   874
Injuries                              787
Injuries Description                  764
Damage ($Mil)                         864
Damage Description                    640
Houses Destroyed                      838
Ho

Taking a look at the number of missing values per column (NaNs) :
- 'Search Parameters' only has one row (the first one) which is not a NaN, this confirms that this column is useless and an artefact.
- Columns with 1 NaN in reality have none as we already confirmed the first row is an artefact.
- Temporal Precision Issues: While Year is almost complete (only the first row is a NaN), Mo (Month) and Dy (Day) have significantly more missing values (132 and 192 respectively). This is expected as historical volcanic records are often imprecise. We know when an eruption happened by year, but not always the exact date. We cannot rely on creating a precise datetime object for every event as on one hand the negative years are note handled by classic Date Time variables and on the other hand setting the default value for dates with missing months and days to the 1st of January would skew our data. For now the Year column is our most reliable temporal variable.
- Sparse Tsunami and Earthquake Data: The Tsu (Tsunami) and Eq (Earthquake) columns are missing in the vast majority of rows (706 and 809 out of 888). In this context, NaN does not mean "unknown", it almost certainly functions as a Boolean False. It indicates that no tsunami or earthquake was recorded for that event as tsunamis and earthquakes are easily recorded.

We verify if the tsunami column is a boolean column

In [ ]:
df_raw['Tsu'].unique()

array([      nan, 3.000e+00, 3.473e+03, 2.100e+01, 3.093e+03, 3.900e+01,
       6.100e+01, 3.102e+03, 2.852e+03, 3.107e+03, 3.112e+03, 2.874e+03,
       5.863e+03, 2.190e+02, 2.220e+02, 3.129e+03, 2.530e+02, 5.754e+03,
       2.690e+02, 2.850e+02, 5.482e+03, 2.930e+02, 3.120e+02, 3.138e+03,
       3.470e+02, 3.141e+03, 3.750e+02, 3.148e+03, 3.150e+03, 3.870e+02,
       4.190e+02, 4.310e+02, 2.853e+03, 5.000e+02, 5.040e+02, 5.250e+02,
       5.280e+02, 5.290e+02, 5.680e+02, 5.770e+02, 3.221e+03, 6.130e+02,
       6.230e+02, 6.310e+02, 6.540e+02, 6.630e+02, 7.060e+02, 7.110e+02,
       7.180e+02, 7.140e+02, 7.300e+02, 7.310e+02, 7.410e+02, 8.090e+02,
       8.210e+02, 8.590e+02, 8.620e+02, 8.690e+02, 9.290e+02, 9.580e+02,
       9.710e+02, 3.451e+03, 1.022e+03, 1.020e+03, 1.068e+03, 1.086e+03,
       1.095e+03, 1.096e+03, 1.109e+03, 5.480e+03, 1.142e+03, 1.143e+03,
       1.144e+03, 1.145e+03, 5.777e+03, 1.175e+03, 1.181e+03, 1.197e+03,
       5.608e+03, 5.629e+03, 1.280e+03, 2.881e+03, 

In [30]:
df_raw['Tsu'].nunique()==df_raw['Tsu'].notna().sum()

np.True_

This is weird, the Tsunami column is filled with incredibly high values. Whatmore there is a different value per Tsunami.

This can be explained by the fact that these values are foreign keys, linking these tsunamis to a tsunami event dataset, this would explain the reason that tsunamis and earthquakes are seperated from the agents column.

We verify that this is also the case for the earthquakes column:

In [28]:
df_raw['Eq'].unique()

array([   nan,  5877.,  7795.,    58.,  7341.,   421.,  9973.,  9788.,
         814.,   920.,  9756.,   942., 10519.,  6879.,  1017.,  1057.,
        8202.,  1129.,  1178.,  1205.,  1304.,  1328.,  1348.,  9532.,
        1515.,  6650.,  6651.,  1787.,  6050.,  1865.,  1867.,  1995.,
        6125., 10184.,  2049.,  6150.,  2125.,  7967.,  2179., 10023.,
        6195.,  6625.,  6198.,  2347.,  6309.,  2595., 10011.,  8158.,
        8391.,  2973.,  2994.,  6877., 10435., 10010.,  3974.,  4116.,
        4227.,  4292.,  4704.,  4876.,  7221.,  4990.,  6652.,  5150.,
        5292.,  5387., 10747.,  6499.,  5568.,  5659., 10555.,  9172.,
       10140., 10332., 10556., 10382., 10383., 10501., 10545., 10744.])

In [29]:
df_raw['Eq'].nunique()==df_raw['Eq'].notna().sum()

np.True_

Since these two columns are foreign keys, we will treat them as boolean values, as in if a row is not null, it is equal to 1, otherwise it is equal to 0.

In [ ]:
def process_tsu_eq_columns(df):
    """
    Replaces NaN values with 0 and non-NaN values with 1 in the 'Tsu' and 'Eq' columns.
    Input : df (pandas.DataFrame)
    Output : df_processed (pandas.DataFrame)
    """
    df_processed = df.copy()
    for col in ['Tsu', 'Eq']:
        if col in df_processed.columns:
            df_processed[col] = df_processed[col].notna().astype(int)
    return df_processed


0      False
1      False
2      False
3      False
4      False
       ...  
883    False
884    False
885    False
886     True
887    False
Name: Tsu, Length: 888, dtype: bool

In [44]:
def clean_data(df):
    """
    Applies preprocessing: renaming columns, handling types, and filling NaNs.
    Input: df (pd.DataFrame) : The raw dataframe
    Output: df (pd.DataFrame) : The cleaned dataframe
    """

    df = df.drop('Search Parameters', axis=1)
    df = df.iloc[1:]
    df = divide_agents(df)
    df = impute_missing_numericals(df)
    df = process_tsu_eq_columns(df)
    
    #Rename columns to remove spaces and special characters for easier coding 
    #and for a better understanding
    df = df.rename(columns={
        'Mo': 'Month',
        'Dy': 'Day',
        'Tsu': 'Tsunami',
        'Eq': 'Earthquake',
        'Damage ($Mil)': 'Damage_Millions',
        'Total Damage ($Mil)': 'Total_Damage_Millions',
        'Elevation (m)': 'Elevation',
        'Total Deaths': 'Total_Deaths',
        'Total Injuries': 'Total_Injuries'
    })

    #If 'Deaths' is empty, we assume 0 for the sake of calculation, rather than dropping the row.
    cols_to_fix = ['VEI', 'Deaths', 'Damage_Millions', 'Injuries']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    #We drop rows without Lat/Lon/Country because we cannot visualize them on a map.
    df = df.dropna(subset=['Latitude', 'Longitude', 'Country'])

    return df

df_cleaned = clean_data(df_raw.copy())
print("Data cleaned.")
df_cleaned.head(5)

--- Imputation Report ---
[Deaths] Filled 127 missing values using calculated medians: {1.0: 2.0, 2.0: 68.0, 3.0: 215.0, 4.0: 2000.0}
[Missing] Filled 4 missing values using calculated medians: {1.0: 9.0, 2.0: 78.0, 3.0: 174.5, 4.0: 1627.5}
[Injuries] Filled 24 missing values using calculated medians: {1.0: 5.0, 2.0: 65.0, 3.0: 277.0, 4.0: 5986.0}
[Damage ($Mil)] Filled 224 missing values using calculated medians: {1.0: 0.52, 2.0: 2.75, 3.0: 15.0, 4.0: 325.0}
[Houses Destroyed] Filled 78 missing values using calculated medians: {1.0: 10.0, 2.0: 63.0, 3.0: 300.0, 4.0: 3085.0}
[Total Deaths] Filled 131 missing values using calculated medians: {1.0: 2.5, 2.0: 68.0, 3.0: 220.5, 4.0: 2000.0}
[Total Missing] Filled 4 missing values using calculated medians: {1.0: 9.5, 2.0: 78.0, 3.0: 174.5, 4.0: 1627.5}
[Total Injuries] Filled 30 missing values using calculated medians: {1.0: 6.5, 2.0: 64.5, 3.0: 250.0, 4.0: 10000.0}
[Total Damage ($Mil)] Filled 241 missing values using calculated medians: {

,Year,Month,Day,Tsunami,Earthquake,Name,Location,Country,Latitude,Longitude,Elevation,Type,VEI,Deaths,Death Description,Missing,Missing Description,Injuries,Injuries Description,Damage_Millions,Damage Description,Houses Destroyed,Houses Destroyed Description,Total_Deaths,Total Death Description,Total Missing,Total Missing Description,Total_Injuries,Total Injuries Description,Total_Damage_Millions,Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description,Agent_?,Agent_A,Agent_E,Agent_F,Agent_G,Agent_I,Agent_L,Agent_M,Agent_P,Agent_S,Agent_T,Agent_W,Agent_m
1,-4360.0,NaN,NaN,0,0,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,0.0,NaN,NaN,NaN,0.0,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0
2,-4350.0,NaN,NaN,0,0,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,215.0,3.0,NaN,NaN,0.0,NaN,15.00,3.0,300.0,3.0,220.5,3.0,NaN,NaN,NaN,NaN,15.0,3.0,300.0,3.0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,-4050.0,NaN,NaN,0,0,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,0.0,NaN,NaN,NaN,0.0,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0
4,-4000.0,NaN,NaN,0,0,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,2.0,1.0,NaN,NaN,0.0,NaN,0.52,1.0,NaN,NaN,2.5,1.0,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,1,0,0
5,-3580.0,NaN,NaN,0,0,Taal,Luzon-Philippines,Philippines,14.002,120.993,311.0,Stratovolcano,6.0,0.0,NaN,NaN,NaN,0.0,NaN,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0


Now we want to see if the shape changed after the cleaning.

In [41]:
show_structure(df_cleaned)

Number of Rows: 887
Number of Columns: 47
Data Types:
Search Parameters                      object
Year                                  float64
Month                                 float64
Day                                   float64
Tsunami                                 int64
Earthquake                              int64
Name                                   object
Location                               object
Country                                object
Latitude                              float64
Longitude                             float64
Elevation                             float64
Type                                   object
VEI                                   float64
Deaths                                float64
Death Description                     float64
Missing                               float64
Missing Description                   float64
Injuries                              float64
Injuries Description                  float64
Damage_Millions           

In [42]:
show_missing_data(df_cleaned)

Missing Data Points per Column:
Search Parameters                     887
Month                                 131
Day                                   191
Death Description                     317
Missing                               871
Missing Description                   873
Injuries Description                  763
Damage Description                    639
Houses Destroyed                      759
Houses Destroyed Description          759
Total_Deaths                          291
Total Death Description               291
Total Missing                         871
Total Missing Description             872
Total_Injuries                        753
Total Injuries Description            753
Total_Damage_Millions                 618
Total Damage Description              618
Total Houses Destroyed                732
Total Houses Destroyed Description    732
dtype: int64


In [53]:
def show_rankings(df):
    """
    Shows the ranking (correlation) between numerical variables.
    Input: df (pd.DataFrame)
    """
    #We select only numeric columns for correlation analysis
    numeric_df = df.select_dtypes(include=[np.number])

    corr_matrix = numeric_df.corr()

    #Let's see what correlates most strongly with Deaths
    if 'Deaths' in corr_matrix.columns:
        print("Ranking of variables correlated with 'Deaths':")
        print(corr_matrix['Deaths'].sort_values(ascending=False))

show_rankings(df_cleaned)

Ranking of variables correlated with 'Deaths':
Deaths                                1.000000
Missing                               0.781042
Total Missing                         0.780995
Total Missing Description             0.766360
Missing Description                   0.760095
Total_Deaths                          0.698954
Injuries                              0.483677
Injuries Description                  0.457936
Houses Destroyed                      0.408233
Total Injuries Description            0.401144
Death Description                     0.394578
Total Death Description               0.362793
Total Houses Destroyed                0.362497
Total_Injuries                        0.358843
Houses Destroyed Description          0.344639
Damage Description                    0.310915
Total Houses Destroyed Description    0.301566
Total Damage Description              0.257396
VEI                                   0.152535
Agent_P                               0.148011
Agent_M      

In [9]:
def investigate_descriptions(df):
    """
    Investigates the content of 'Description' columns to understand why they are numeric
    and how they relate to the absolute counts (like Total_Deaths).
    
    Input: df (pd.DataFrame)
    """
    print("Inspecting Unique Values in Description Columns")
    # We suspect these are discrete classes (1, 2, 3, 4), not random numbers.
    # Let's check the unique values to confirm.
    desc_cols = ['Death Description', 'Damage Description']
    
    for col in desc_cols:
        if col in df.columns:
            unique_vals = sorted(df[col].dropna().unique())
            print(f"Unique values in '{col}': {unique_vals}")
    
    print()

    print("Testing Hypothesis: Is 'Death Description' a proxy for 'Total_Deaths'?")
    
    # We will group the data by 'Death Description' and see the range of 'Total_Deaths'
    if 'Total_Deaths' in df.columns and 'Death Description' in df.columns:
        
        # Group by the description class and calculate stats for the actual death count
        description_stats = df.groupby('Death Description')['Total_Deaths'].agg(
            Count='count', 
            Min_Deaths='min', 
            Median_Deaths='median', 
            Max_Deaths='max',
            Mean_Deaths='mean'
        )
        print(description_stats)
    
    print()
    
    #Correlation Check
    if 'Total_Deaths' in df.columns and 'Death Description' in df.columns:
        corr = df[['Total_Deaths', 'Death Description']].corr().iloc[0,1]
        print(f"Correlation between 'Total_Deaths' and 'Death Description': {corr:.4f}")

investigate_descriptions(df_cleaned)

Inspecting Unique Values in Description Columns
Unique values in 'Death Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Damage Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]

Testing Hypothesis: Is 'Death Description' a proxy for 'Total_Deaths'?
                   Count  Min_Deaths  Median_Deaths  Max_Deaths  Mean_Deaths
Death Description                                                           
1.0                  336         1.0            3.0     15000.0    52.375000
2.0                   26        54.0           68.0       226.0    77.615385
3.0                   43       106.0          215.0      1000.0   321.860465
4.0                   38      1001.0         2000.0     60000.0  7042.894737

Correlation between 'Total_Deaths' and 'Death Description': 0.3785


In [10]:
import plotly.express as px

def visualize_description_vs_deaths(df):
    """
    Creates a box plot to visually compare the 'Death Description' categories
    against the actual 'Total_Deaths' numbers.
    """
    plot_df = df.dropna(subset=['Death Description', 'Total_Deaths']).copy()
    
    # Convert Description to string so it's treated as a category (Class 1, Class 2...), not a continuous number
    plot_df['Category'] = plot_df['Death Description'].astype(str)
    
    fig = px.box(
        plot_df, 
        x='Category', 
        y='Total_Deaths',
        title="Distribution of Deaths by Death Description Category",
        labels={'Category': 'Death Description Class (1-4)', 'Total_Deaths': 'Confirmed Deaths (Log Scale)'},
        category_orders={'Category': ["1.0", "2.0", "3.0", "4.0"]}, # Order the bars logically
        log_y=True, # Use Log scale because volcanic deaths vary wildly (from 1 to 30,000)
        points="all", 
        template='plotly_white'
    )
    
    fig.show()

visualize_description_vs_deaths(df_cleaned)

In [11]:
def analyze_all_description_pairs(df):
    """
    Loops through multiple metric/description pairs to verify if they all follow 
    the ordinal scale pattern (1-4).
    """
    # We want to investigate the pairs as followed :  (Metric Column, Description Column)
    pairs_to_check = [
        ('Missing', 'Missing Description'),
        ('Injuries', 'Injuries Description'),
        ('Houses Destroyed', 'Houses Destroyed Description'),
        ('Total_Damage_Millions', 'Total Damage Description') 
    ]

    for metric_col, desc_col in pairs_to_check:
        
        # Check if columns exist in the dataframe before proceeding
        if metric_col in df.columns and desc_col in df.columns:
            print(f"Analysis: {metric_col}  vs.  {desc_col}")

            
            #Statistical Check
            # Group by the description (1, 2, 3, 4) and see the range of the actual numbers
            stats = df.groupby(desc_col)[metric_col].agg(
                Count='count', 
                Min='min', 
                Median='median', 
                Max='max',
                Mean='mean'
            )
            print(stats)
            
            # Filter data for plotting
            plot_df = df.dropna(subset=[metric_col, desc_col]).copy()
            plot_df['Category'] = plot_df[desc_col].astype(str) # Convert to string for categorical plotting
            
            fig = px.box(
                plot_df, 
                x='Category', 
                y=metric_col,
                title=f"Distribution of {metric_col} by {desc_col}",
                labels={'Category': 'Severity Class (1-4)', metric_col: f'{metric_col} (Log Scale)'},
                category_orders={'Category': ["1.0", "2.0", "3.0", "4.0"]}, 
                log_y=True, # Log scale is crucial because values vary from 1 to Millions
                points="all", 
                template='plotly_white'
            )
            fig.show()
            
        else:
            print(f"Skipping pair {metric_col}/{desc_col}: Columns not found.")

analyze_all_description_pairs(df_cleaned)

Analysis: Missing  vs.  Missing Description
                     Count     Min  Median     Max    Mean
Missing Description                                       
1.0                      5     2.0     9.0    44.0    15.6
2.0                      1    78.0    78.0    78.0    78.0
3.0                      2   120.0   174.5   229.0   174.5
4.0                      2  1500.0  1627.5  1755.0  1627.5


Analysis: Injuries  vs.  Injuries Description
                      Count  Min  Median      Max         Mean
Injuries Description                                          
1.0                      93  0.0     4.0     50.0     9.526882
2.0                      12  0.0    53.0     86.0    37.833333
3.0                      16  0.0   203.0   1000.0   242.375000
4.0                       3  0.0  1972.0  10000.0  3990.666667


Analysis: Houses Destroyed  vs.  Houses Destroyed Description
                              Count     Min  Median     Max         Mean
Houses Destroyed Description                                            
1.0                              19     1.0    10.0    75.0    18.894737
2.0                               5    60.0    63.0    90.0    71.000000
3.0                              14   144.0   300.0   800.0   377.428571
4.0                              12  1109.0  3085.0  9000.0  3593.166667


Analysis: Total_Damage_Millions  vs.  Total Damage Description
                          Count   Min   Median     Max        Mean
Total Damage Description                                          
1.0                           1   1.0    1.000     1.0    1.000000
2.0                           8   2.0    2.750     4.0    2.908000
3.0                           7   5.0   15.000    20.0   14.714286
4.0                          12  67.0  192.684  2000.0  424.364000


Based on the NOAA Significant Volcanic Eruption Database and the classification standards by Simkin and Siebert (1994), the Agent column indicates the specific volcanic hazard or phenomenon that caused fatalities, injuries, or damage during an eruption.
P: Pyroclastic flow or surge (a fast-moving current of hot gas and volcanic matter).

M: Mudflow or Lahar (volcanic mudflow).

T: Tsunami (generated by the eruption).

L: Lava flow.

G: Gas (toxic volcanic gases).

F: Tephra/Ash fall (falling volcanic rock and ash).

A: Avalanche (debris avalanche or landslide).

E: Electrical (lightning associated with the eruption).

I: Indirect causes (such as starvation, disease, or exposure resulting from the eruption).

S: Seismic activity (earthquakes related to the eruption).

### Interpretation of Missing Data & Variable Selection

**1. Observations on Data Quality:**
Our analysis reveals a significant amount of missing data across the dataset. However, a deeper inspection allows us to categorize these gaps into two types:
* Irrelevant/Noise: Columns like Search Parameters (887 missing) are largely empty and do not contain information useful for our visualization goals.
* Redundant Proxies: Columns like Death Description or Damage Description appeared at first to be sparse floats. Our investigation reveals these are actually Ordinal Severity Scales (ranked 1 to 4) rather than continuous measurements.
* We are also going to drop Month and Day because the exact date is not necessary and relevant especially since a part of the dataset doesn't have this data.

**2. Decision on "Description" Columns:**
* We confirmed a high correlation between Death Description and Total_Deaths. For example, a "Description" value of 4 consistently corresponds to catastrophic events with high death tolls.
* We will exclude these description columns from our final dashboard dataset.
* They are redundant. Since we have the precise absolute numbers in Total_Deaths and Total_Damage_Millions, keeping the simplified 1-4 scale would introduce multicollinearity (repetition) without adding precision. We prioritize the exact figures for clearer visualizations ("5,000 deaths" is more informative to a user than "Severity Level 3").

**3. Decision on Flags :**
* Columns like Tsunami and Earthquake are "flags" often left blank when the event did not occur.
We will keep them in the Dataframe and analyze later if they can be useful to our study

**4. Statistical Caution (Means vs. Totals):**
* Because missing values in impact columns (Total_Deaths) are frequent and likely represent "zero" or "unknown" in historical contexts, calculating an average would be misleading.
We will avoid using the mean as a central KPI. Instead, we will use Sums (Total Global Deaths) and Counts (Number of Eruptions) which remain robust even with sparse historical records.

In [12]:
def remove_unnecessary_columns(df):
    """
    Removes columns deemed irrelevant, sparse, or redundant for the visualization dashboard.
    
    Why we are doing this:
    'Search Parameters': Contains metadata about the database query, not the volcano itself.
    'Month' & 'Day': Too granular. We are analyzing historical trends over centuries (by Year), 
    so specific dates are noise for this high-level overview.
    'Description' Columns : We proved these are Ordinal Scales (1-4)
    that are redundant because we already have the precise metrics (e.g., 'Total_Deaths').
    Removing them prevents multicollinearity and cleans the dataset.
       
    Input: df (pd.DataFrame)
    Output: df (pd.DataFrame) - The reduced and final dataframe.
    """
    cols_to_drop = ['Search Parameters', 'Month', 'Day']
    
    #Dynamically find all 'Description' columns
    description_cols = [col for col in df.columns if 'Description' in col]
    
    all_cols_to_remove = cols_to_drop + description_cols
    
    # errors='ignore' ensures the code doesn't crash if we accidentally run it twice (and columns are already gone)
    df = df.drop(columns=all_cols_to_remove, errors='ignore')
    
    print(f"Removed {len(all_cols_to_remove)} columns: {all_cols_to_remove}")
    print(f"Remaining columns: {df.columns.tolist()}")
    
    return df

df_final = remove_unnecessary_columns(df_cleaned)
df_final.head()

Removed 13 columns: ['Search Parameters', 'Month', 'Day', 'Death Description', 'Missing Description', 'Injuries Description', 'Damage Description', 'Houses Destroyed Description', 'Total Death Description', 'Total Missing Description', 'Total Injuries Description', 'Total Damage Description', 'Total Houses Destroyed Description']
Remaining columns: ['Year', 'Tsunami', 'Earthquake', 'Name', 'Location', 'Country', 'Latitude', 'Longitude', 'Elevation', 'Type', 'VEI', 'Agent', 'Deaths', 'Missing', 'Injuries', 'Damage_Millions', 'Houses Destroyed', 'Total_Deaths', 'Total Missing', 'Total_Injuries', 'Total_Damage_Millions', 'Total Houses Destroyed']


,Year,Tsunami,Earthquake,Name,Location,Country,Latitude,Longitude,Elevation,Type,VEI,Agent,Deaths,Missing,Injuries,Damage_Millions,Houses Destroyed,Total_Deaths,Total Missing,Total_Injuries,Total_Damage_Millions,Total Houses Destroyed
1,-4360.0,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,-4350.0,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,P,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,-4050.0,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,-4000.0,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,T,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
5,-3580.0,NaN,NaN,Taal,Luzon-Philippines,Philippines,14.002,120.993,311.0,Stratovolcano,6.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Load data for notebook analysis (outside of the dashboard)
df = load_raw_data('volcano-events.tsv')
df.head()

File loaded.


,Search Parameters,Year,Mo,Dy,Tsu,Eq,Name,Location,Country,Latitude,Longitude,Elevation (m),Type,VEI,Agent,Deaths,Death Description,Missing,Missing Description,Injuries,Injuries Description,Damage ($Mil),Damage Description,Houses Destroyed,Houses Destroyed Description,Total Deaths,Total Death Description,Total Missing,Total Missing Description,Total Injuries,Total Injuries Description,Total Damage ($Mil),Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description
0,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,-4360.0,NaN,NaN,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,-4350.0,NaN,NaN,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,P,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
3,NaN,-4050.0,NaN,NaN,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-4000.0,NaN,NaN,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,T,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN


## Spatial Analysis

In [14]:
# Function: Build a scatter-based world map showing individual volcano eruptions.
# Input:
#   - df (pd.DataFrame): filtered volcano dataset containing at least
#       ["Latitude", "Longitude", "Type", "Name", "VEI", "Country", "Year", "Deaths"]
# Output:
#   - fig (plotly.graph_objs.Figure): an interactive scatter_geo map (used in Dash)

def get_map_points(df):
    # Copy the dataframe to avoid modifying the original one
    df_map = df.copy()

    # Replace missing VEI values with a small default value (0.5)
    # to avoid invisible points on the map
    df_map['VEI_Size'] = df_map['VEI'].fillna(0.5)

    # Create the base geographic scatter plot
    fig = px.scatter_geo(
        df_map,
        lat="Latitude",           # latitude of volcano
        lon="Longitude",          # longitude of volcano
        color="Type",             # volcano morphological category
        size="VEI_Size",          # bubble size based on VEI (explosivity)
        hover_name="Name",        # volcano name in tooltip
        hover_data={              # additional tooltip information
            "Country": True,
            "Year": True,
            "Deaths": True,
            "VEI": True,
            "VEI_Size": False
        },
        title="Global Volcano Distribution (Bubble size = VEI)",
        projection="natural earth", # projection style
        size_max=15,                # maximum bubble size
        template="plotly_dark"      # dark theme to match dashboard
    )

    # Custom color palette: 20 vivid volcanic colors (orange → red → magenta → violet)
    warm_palette = [
        "#ffb74d", "#ffa726", "#ff9800", "#fb8c00", "#f57c00", "#ef6c00",
        "#e65100", "#ff6d00", "#ff3d00", "#dd2c00",
        "#ff1744", "#f50057", "#d50000", "#c51162", "#aa00ff",
        "#9c27b0", "#8e24aa", "#7b1fa2", "#6a1b9a", "#6200ea"
    ]

    n_traces = len(fig.data)   # one trace per volcano Type
    n_colors = len(warm_palette)

    # Assign a distinct color from the palette to each volcano Type
    for i, trace in enumerate(fig.data):
        color = warm_palette[i % n_colors]     # loop through palette if Types > 20
        trace.marker.update(
            color=color,
            line=dict(width=0)                 # remove outline for cleaner look
        )

    # Add geographic features (coastlines, countries, land)
    fig.update_geos(
        showcountries=True,
        showcoastlines=True,
        showland=True,
    )

    # Transparent background to match the dashboard's dark theme
    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )

    return fig

In [15]:
# Function: Build a choropleth (colored world map) aggregated by country.
# Input:
#   - df (pd.DataFrame): filtered volcano dataset containing at least:
#       ["Country", value_col]
#   - value_col (str): column name used for coloring (e.g., "Count", "Deaths", "Damage_Millions")
#   - title (str): title of the choropleth
#   - color_label (str): label for the colorbar (e.g., "Eruptions", "Deaths", "Damage")
# Output:
#   - fig (plotly.graph_objs.Figure): an interactive choropleth map (used in Dash)

def get_country_choropleth(df, value_col, title, color_label):
    # Aggregate data by country for the selected metric (count, deaths, damage…)
    agg = df.groupby('Country', as_index=False)[value_col].sum()
    # Custom continuous volcanic colormap (dark red → orange → magenta → violet)
    # Designed to avoid white/yellow and emphasize bright volcanic tones
    volcano_scale = [
        (0.00, "#000000"),   # very dark base
        (0.05, "#4b0000"),   # deep red
        (0.10, "#7f0000"),   # darker red
        (0.20, "#b00000"),   # intense red
        (0.30, "#d50000"),   # bright red
        (0.40, "#ff1400"),   # red-orange (flashy)
        (0.55, "#ff3d00"),   # bright orange-red
        (0.70, "#ff6d00"),   # orange incandescent
        (0.85, "#ff8500"),   # bright orange
        (0.93, "#d81b60"),   # magenta
        (1.00, "#6a1b9a")    # deep violet
    ]
    # Build the choropleth map
    fig = px.choropleth(
        agg,
        locations='Country',            # country name column
        locationmode='country names',   # match names to world countries
        color=value_col,                # metric used for color intensity
        hover_name='Country',           # tooltip title
        title=title,
        labels={value_col: color_label}, # name of the color axis
        template="plotly_dark",          # dark theme
        color_continuous_scale=volcano_scale
    )
    # Display country borders, coastlines, and land
    fig.update_geos(
        showcountries=True,
        showcoastlines=True,
        showland=True
    )
    # Transparent background to blend with the dashboard
    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )

    return fig


In [29]:

def compare_death_columns(file_path='volcano-events.tsv'):
    # 1. Load Data
    try:
        df = pd.read_csv(file_path, sep='\t')
    except FileNotFoundError:
        return "File not found. Please ensure 'volcano-events.tsv' is in the folder."

    # 2. Select and Clean Columns
    # Create a subset to work with
    comparison_df = df[['Name', 'Year', 'Deaths', 'Total Deaths']].copy()
    
    # Rename for easier access
    comparison_df.rename(columns={'Total Deaths': 'Total_Deaths'}, inplace=True)

    # Convert to numeric, coercing errors to NaN, then filling with 0 for comparison
    comparison_df['Deaths'] = pd.to_numeric(comparison_df['Deaths'], errors='coerce').fillna(0)
    comparison_df['Total_Deaths'] = pd.to_numeric(comparison_df['Total_Deaths'], errors='coerce').fillna(0)

    # 3. Analyze Differences
    # Create a 'Difference' column
    comparison_df['Diff'] = comparison_df['Total_Deaths'] - comparison_df['Deaths']
    
    # Identify rows where they are different (Total > Direct)
    different_rows = comparison_df[comparison_df['Diff'].abs() > 0].copy()
    
    print(f"Total Rows Scanned: {len(comparison_df)}")
    print(f"Events with Discrepancies (Total > Direct): {len(different_rows)}")
    
    if not different_rows.empty:
        print("\nTop 10 Events where Secondary Effects (Tsunami, etc.) increased the Death Toll:")
        print(different_rows.sort_values('Diff', ascending=False).head(10)[['Name', 'Year', 'Deaths', 'Total_Deaths', 'Diff']].to_string(index=False))

    # 4. Visualization
    # Scatter plot with Log Scale (to handle massive outliers like Tambora)
    fig = px.scatter(
        comparison_df, 
        x='Deaths', 
        y='Total_Deaths',
        hover_name='Name',
        hover_data=['Year', 'Diff'],
        title='<b>Impact Analysis</b>: Direct Deaths vs. Total Deaths',
        template='plotly_dark',
        labels={'Deaths': 'Direct Deaths (Immediate)', 'Total_Deaths': 'Total Deaths (Including Secondary)'}
    )
    
    # Add a red dashed line for x=y (Perfect agreement)
    # Points ABOVE this line mean secondary hazards (Tsunamis, Starvation) added to the toll.
    # Points ON this line mean the eruption itself was the only killer.
    max_val = comparison_df['Total_Deaths'].max()
    fig.add_shape(
        type="line", line=dict(dash='dash', color='red', width=2),
        x0=1, y0=1, x1=max_val, y1=max_val # Start at 1 to avoid Log(0) issues visually
    )
    
    fig.update_layout(
        xaxis_type="log", 
        yaxis_type="log",
        annotations=[dict(
            text="Points above red line = <br>Secondary effects (Tsunami/Starvation) <br>caused significantly more deaths.",
            x=0.05, y=0.95, xref='paper', yref='paper', showarrow=False, align="left", bgcolor="rgba(0,0,0,0.5)"
        )]
    )

    return fig

# Run the function
fig = compare_death_columns()
fig.show()

Total Rows Scanned: 888
Events with Discrepancies (Total > Direct): 40

Top 10 Events where Secondary Effects (Tsunami, etc.) increased the Death Toll:
               Name   Year  Deaths  Total_Deaths    Diff
            Tambora 1815.0 11000.0       60000.0 49000.0
           Krakatau 1883.0  2000.0       36417.0 34417.0
               Etna 1169.0     0.0       16000.0 16000.0
          Unzendake 1792.0     0.0       15000.0 15000.0
          Grimsvotn 1784.0     0.0        9350.0  9350.0
        Santa Maria 1902.0  2500.0       10000.0  7500.0
      Oshima-Oshima 1741.0     0.0        2000.0  2000.0
          Iliwerung 1979.0     0.0        1239.0  1239.0
          Sao Jorge 1757.0     0.0        1053.0  1053.0
Hokkaido-Komagatake 1640.0     0.0         700.0   700.0


In [ ]:

df['Deaths_Clean'] = pd.to_numeric(df['Deaths'], errors='coerce').fillna(0)
df['Total_Deaths_Clean'] = pd.to_numeric(df['Total_Deaths'], errors='coerce').fillna(0)

# 3. Filter for the condition: Total < Deaths
# We verify both are non-zero to find logical errors, or remove the second condition to find missing totals.
anomalous_rows = df[
    (df['Total_Deaths_Clean'] < df['Deaths_Clean']) 
]

# 4. Display the results
print(f"Found {len(anomalous_rows)} anomalous rows.")
display_columns = ['Name', 'Year', 'Deaths', 'Total_Deaths', 'Country']
print(anomalous_rows[display_columns])

Found 4 anomalous rows.
                Name    Year  Deaths  Total_Deaths        Country
585    Hudson, Cerro  1971.0     5.0           3.0          Chile
736          Kilauea  1998.0     1.0           NaN  United States
761  Tengger Caldera  2004.0     2.0           NaN      Indonesia
872           Semeru  2021.0    51.0          45.0      Indonesia


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[27], line 198, in update_dashboard(
    selected_country=None,
    year_range=[-4360, 2024],
    map_mode='points'
)
    195 total_damage = f"${dff['Damage_Millions'].sum():,.0f}"
    197 # Main figures
--> 198 fig_map = get_map_figure(dff)
        dff =     Search Parameters    Year  Month   Day  Tsunami  Earthquake       Name  \
1                 NaN -4360.0    NaN   NaN      NaN         NaN   Macauley   
2                 NaN -4350.0    NaN   NaN      NaN         NaN      Kikai   
3                 NaN -4050.0    NaN   NaN      NaN         NaN     Masaya   
4                 NaN -4000.0    NaN   NaN      NaN         NaN     Witori   
5                 NaN -3580.0    NaN   NaN      NaN         NaN       Taal   
..                ...     ...    ...   ...      ...         ...        ...   
883               NaN  2024.0    4.0  2

## Temporal Analysis of volcanic Eruptions

In [16]:
# ============================
# 1. Yearly aggregation helper
# ============================

def build_eruptions_per_year(df):
    """
    Build a yearly time series: index = Year, value = number of eruptions.
    We drop missing or invalid (<= 0) years.
    """
    temp = df.dropna(subset=["Year"]).copy()
    temp = temp[temp["Year"] > 0]

    eruptions_per_year = (
        temp
        .groupby("Year")
        .size()
        .sort_index()
    )

    eruptions_per_year.index = eruptions_per_year.index.astype(int)
    eruptions_per_year.name = "eruptions_per_year"

    return eruptions_per_year


In [17]:
# =====================================
# 2. Aggregation by year or by century
# =====================================

def aggregate_eruptions(df, by="year"):
    """
    Aggregate eruptions by year or by century.
    Returns a dataframe with two columns: period, count.
    """
    temp = df.dropna(subset=["Year"]).copy()
    temp = temp[temp["Year"] > 0]  # keep AD years only

    if by == "year":
        grouped = (
            temp.groupby("Year")
            .size()
            .reset_index(name="count")
        )
        grouped.rename(columns={"Year": "period"}, inplace=True)

    elif by == "century":
        # Century numbering: 1..n (e.g. 19 = 1801-1900)
        temp["century"] = ((temp["Year"] - 1) // 100 + 1).astype(int)
        grouped = (
            temp.groupby("century")
            .size()
            .reset_index(name="count")
        )
        grouped.rename(columns={"century": "period"}, inplace=True)

    else:
        raise ValueError("by must be 'year' or 'century'")

    return grouped

In [18]:
# ====================================
# 3. Plotly time series (year/century)
# ====================================

def render_time_series(df, by="year"):
    """
    Build a time-series figure of eruption frequency over time
    using Plotly Express, either per year or per century.
    """
    agg = aggregate_eruptions(df, by=by)

    if by == "year":
        x_label = "Year"
        title = "Number of eruptions per year"
    else:
        x_label = "Century"
        title = "Number of eruptions per century"

    fig = px.line(
        agg,
        x="period",
        y="count",
        markers=True,
        labels={"period": x_label, "count": "Number of eruptions"},
        title=title,
        color_discrete_sequence=['#ff5722']  # orange/red curve
    )

    fig.update_layout(
        height=320,
        paper_bgcolor='rgba(0,0,0,0)',   # transparent to fit the dark card
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white"),
        xaxis=dict(
            tickmode="auto",
            tickangle=-45
        ),
        margin=dict(l=50, r=20, t=60, b=40)
    )

    fig.update_traces(
        hovertemplate=f"{x_label}: %{{x}}<br>Eruptions: %{{y}}<extra></extra>"
    )

    return fig

In [19]:
# =====================================================
# 4. Time series mode switcher for the dashboard (year /
#    century / smoothed yearly series with moving average)
# =====================================================

def render_temporal_series(df, mode="year"):
    """
    3 modes for the dashboard:
    - 'year'    : time series per year (uses render_time_series)
    - 'century' : time series per century (uses render_time_series)
    - 'smooth'  : yearly series + 10-year moving average (Plotly)
    """
    # 1) Yearly series (raw)
    if mode == "year":
        return render_time_series(df, by="year")

    # 2) Century series (raw)
    if mode == "century":
        return render_time_series(df, by="century")

    # 3) Smoothed yearly series (10-year moving average)
    if mode == "smooth":
        per_year = build_eruptions_per_year(df)
        smooth = per_year.rolling(window=10).mean()

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=per_year.index,
            y=per_year.values,
            mode="lines",
            name="Raw yearly data",
            line=dict(width=1, color="#888")   # grey, less dominant
        ))
        fig.add_trace(go.Scatter(
            x=smooth.index,
            y=smooth.values,
            mode="lines",
            name="10-year moving average",
            line=dict(width=3, color="#ff5722")  # main orange/red curve
        ))

        fig.update_layout(
            title="Smoothed eruptions per year (10-year moving average)",
            xaxis_title="Year",
            yaxis_title="Number of eruptions",
            height=320,
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
            font=dict(color="white"),
            margin=dict(l=50, r=20, t=60, b=40)
        )

        fig.update_traces(
            hovertemplate="Year: %{x}<br>Eruptions: %{y}<extra></extra>"
        )

        return fig

    # Fallback (safety)
    return render_time_series(df, by="year")

In [20]:
# =========================
# 5. Prophet data preparation
# =========================

def prepare_prophet_df(eruptions_per_year, min_year=1800):
    """
    Convert the yearly eruption series into Prophet's expected format.
    Keep only modern years (>= min_year) for a cleaner and valid forecast.
    """
    # Series -> DataFrame
    df_prophet = eruptions_per_year.reset_index()
    df_prophet.columns = ["Year", "y"]

    # Keep only modern years (for datetime + data quality reasons)
    df_prophet = df_prophet[df_prophet["Year"] >= min_year]

    # Create a datetime column 'ds' (required by Prophet)
    df_prophet["ds"] = pd.to_datetime(
        df_prophet["Year"].astype(int).astype(str) + "-01-01"
    )

    return df_prophet[["ds", "y"]]

Restricting the time range for forecasting

The original volcanic eruption dataset spans several millennia, including
very early years (e.g. 200 AD). However, pandas datetime objects and the
Prophet model are not designed to handle such ancient dates directly.

To build a meaningful and technically valid forecast, we restrict the
time range to **modern years** (e.g. from 1800 onwards). This also makes
sense from a data quality perspective: recent centuries have much more
complete and reliable recording of volcanic activity.

The forecasting model is therefore fitted only on this modern subset of
the time series, and the predictions should be interpreted within this
context.


In [21]:
# =====================
# 6. Prophet forecasting
# =====================

def forecast_eruptions(df_prophet, n_future_years=20):
    """
    Fit a Prophet model on the historical data and forecast n_future_years ahead.
    """
    model = Prophet()
    model.fit(df_prophet)

    # Generate future dates (one point per year)
    future_dates = model.make_future_dataframe(periods=n_future_years, freq="YE")

    # Predict on both historical + future dates
    prediction = model.predict(future_dates)

    return prediction

In [22]:
# ==================================
# 7. Plotly figure for the forecast
# ==================================

def make_forecast_figure(df_prophet, prediction):
    """
    Build a Plotly figure showing historical yearly eruptions
    and Prophet forecast.
    """
    fig = go.Figure()

    # Historical data
    fig.add_trace(go.Scatter(
        x=df_prophet["ds"],
        y=df_prophet["y"],
        mode="lines",
        name="Historical",
        line=dict(color="#ff5722", width=2)
    ))

    # Forecast
    fig.add_trace(go.Scatter(
        x=prediction["ds"],
        y=prediction["yhat"],
        mode="lines",
        name="Forecast",
        line=dict(color="#ff9800", width=2, dash="dash")  # orange, dashed
    ))

    fig.update_layout(
        title="Forecast of volcanic eruptions per year (Prophet)",
        xaxis_title="Year",
        yaxis_title="Number of eruptions",
        height=320,
        paper_bgcolor='rgba(0,0,0,0)',   # same as other cards
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white"),
        margin=dict(l=50, r=20, t=60, b=40),
        legend=dict(bgcolor="rgba(0,0,0,0)")
    )

    return fig


In [23]:
def make_empty_forecast_figure(message):
    """
    Simple empty figure used when there is not enough data
    to fit a reliable Prophet model (e.g. after filters).
    """
    fig = go.Figure()
    fig.update_layout(
        title=message,
        xaxis_title="Time",
        yaxis_title="Number of eruptions",
        height=320,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white"),
        margin=dict(l=50, r=20, t=60, b=40)
    )
    return fig

In [24]:
# Yearly time series (raw)
fig_ts_year = render_time_series(df, by="year")
fig_ts_year.show()

# Time series with moving average (smooth mode)
fig_ts_smooth = render_temporal_series(df, mode="smooth")
fig_ts_smooth.show()

# Forecast
eruptions_per_year = build_eruptions_per_year(df)
df_prophet = prepare_prophet_df(eruptions_per_year, min_year=1800)
prediction = forecast_eruptions(df_prophet, n_future_years=50)
fig_forecast = make_forecast_figure(df_prophet, prediction)
fig_forecast.show()

16:38:58 - cmdstanpy - INFO - Chain [1] start processing
16:38:58 - cmdstanpy - INFO - Chain [1] done processing


## Impact Analysis

In [25]:
def get_impact_figure(df):
    top_deadly = df.nlargest(10, 'Deaths').sort_values('Deaths', ascending=True)
    fig = px.bar(
        top_deadly,
        x="Deaths",
        y="Name",
        orientation='h',
        text="Deaths",
        title="Top 10 Deadliest Eruptions",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722', textposition='outside')
    fig.update_layout(
        xaxis_title="Deaths", 
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Correlation Analysis

In [26]:
def get_correlation_figure(df):
    damage_df = df[df['Damage_Millions'] > 0].copy()
    if damage_df.empty:
         return px.scatter(title="No Data")

    fig = px.scatter(
        damage_df,
        x="VEI",
        y="Damage_Millions",
        size="Deaths",
        hover_name="Name",
        log_y=True,
        title="VEI vs. Impact",
        template="plotly_dark"
    )
    fig.update_traces(marker=dict(color='#ff5722', opacity=0.7))
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## UI for Dashboard

A Visualization Dashboard: Use Python Dash (https://dash.plotly.com/) to visualize a
dashboard containing the four elements built in the previous step. The dashboard should include the
names of the team members and the name of the dataset used for this project and the description of
the project objective. Remember that a goal should be concise, achievable, and tangible.

In [27]:
# Initialize App
app = Dash(__name__)

# Load Data
df = load_raw_data('volcano-events.tsv')
df = clean_data(df)

# Styles
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#111111",
    "color": "white"
}

CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
    "background-color": "#000000",
    "min-height": "100vh",
    "color": "white"
}

CARD_STYLE = {
    "background-color": "#1e1e1e",
    "padding": "20px",
    "border-radius": "10px",
    "margin-bottom": "20px",
    "box-shadow": "0 4px 6px rgba(0,0,0,0.3)"
}

# Layout
app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Volcano Insights", style={'font-size': '20px', 'margin-bottom': '20px', 'color': '#ff5722'}),
        html.Hr(style={'border-color': '#333'}),
        html.P("Filters", style={'color': '#888'}),
        
        html.Label("Year Range", style={'margin-top': '20px'}),
        dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            value=[df['Year'].min(), df['Year'].max()],
            marks={str(year): str(year) for year in range(int(df['Year'].min()), int(df['Year'].max()), 1000)},
            tooltip={"placement": "bottom", "always_visible": True},
            className="dark-slider"
            
        ),
        
        html.Label("Country", style={'margin-top': '20px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country'].unique())],
            placeholder="All Countries",
            style={'color': 'black'}  # dropdown text
        )
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        html.H1("Volcano Insights Dashboard", style={'margin-bottom': '5px'}),
        html.P("Analyzing Significant Volcanic Eruptions", style={'color': '#888', 'margin-bottom': '30px'}),

        # KPI Row
        html.Div([
            html.Div([
                html.H4("Total Eruptions", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-eruptions', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Deaths", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-deaths', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Damage ($M)", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-damage', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

        # Charts Row 1
       html.Div([
    # === BIG MAP CARD ===
    html.Div(
        [
            # Choix du type de carte
            html.Div([
                html.Label("Map view:", style={'color': 'white', 'margin-right': '10px'}),
                dcc.RadioItems(
                    id='map-mode',
                    options=[
                        {'label': 'Eruptions (points)', 'value': 'points'},
                        {'label': 'Eruptions / country', 'value': 'eruptions_country'},
                        {'label': 'Deaths / country', 'value': 'deaths_country'},
                        {'label': 'Damage / country', 'value': 'damage_country'},
                    ],
                    value='points',
                    inline=True,
                    className='dark-radio'
                )
            ], style={'margin-bottom': '10px'}),

            # La figure 
            dcc.Graph(id='map-graph', style={'height': '650px', 'width': '100%'})
        ],
        style={**CARD_STYLE, 'flex': '2', 'margin-right': '20px', 'padding': '10px'}
    ),

    # === CARD DE DROITE (time graph comme avant) ===
    html.Div(
        [dcc.Graph(id='time-graph', style={'height': '650px', 'width': '100%'})],
        style={**CARD_STYLE, 'flex': '1', 'padding': '10px'}
    )
], style={'display': 'flex', 'margin-bottom': '20px'}),

        # Charts Row 2
        html.Div([
            html.Div([dcc.Graph(id='impact-graph')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='corr-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex'}),

        # Charts Row 3 - Temporal Analysis
        html.Div([
            html.H4(
                "Temporal dynamics",
                style={
                    'margin-bottom': '5px',
                    'color': 'white',
                    'font-size': '26px',
                    'font-weight': 'bold',
                    'font-family': '"Open Sans", "Helvetica", "Arial", sans-serif'
                }
            ),

            html.Div([
                # LEFT GRAPH (time series with mode selector)
                html.Div([
                    dcc.RadioItems(
                        id='ts-mode',
                        options=[
                            {'label': 'Per year', 'value': 'year'},
                            {'label': 'Per century', 'value': 'century'},
                            {'label': 'Smoothed (10-year MA)', 'value': 'smooth'},
                        ],
                        value='year',
                        inline=True,
                        style={'margin-bottom': '10px', 'color': 'white'}
                    ),
                    dcc.Graph(id='time-series-graph', style={'height': '330px'})
                ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),

                # RIGHT GRAPH (forecast)
                html.Div([
                    dcc.Graph(id='forecast-year-graph', style={'height': '330px'})
                ], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex'})
        ])
    ], style=CONTENT_STYLE)
])


@app.callback(
    [Output('map-graph', 'figure'),
     Output('time-graph', 'figure'),
     Output('impact-graph', 'figure'),
     Output('corr-graph', 'figure'),
     Output('kpi-eruptions', 'children'),
     Output('kpi-deaths', 'children'),
     Output('kpi-damage', 'children')],
    [Input('country-dropdown', 'value'),
     Input('year-slider', 'value'),
     Input('map-mode', 'value')] 
     
)
def update_dashboard(selected_country, year_range, map_mode):
    dff = df.copy()

    # Filtre pays
    if selected_country:
        dff = dff[dff['Country'] == selected_country]

    # Filtre années
    if year_range:
        dff = dff[(dff['Year'] >= year_range[0]) & (dff['Year'] <= year_range[1])]

    # KPIs
    total_eruptions = len(dff)
    total_deaths = f"{int(dff['Deaths'].sum()):,}"
    total_damage = f"${dff['Damage_Millions'].sum():,.0f}"

    # Main figures
    fig_map = get_map_figure(dff)
    fig_impact = get_impact_figure(dff)
    fig_corr = get_correlation_figure(dff)

    # Temporal analysis: year / century / smooth
    fig_ts = render_temporal_series(dff, mode=ts_mode)

    # Forecast per year (modern period only, e.g. >= 1800)
    eruptions_per_year = build_eruptions_per_year(dff)
    df_prophet_year = prepare_prophet_df(eruptions_per_year, min_year=1800)

    if len(df_prophet_year) > 5:
        prediction_year = forecast_eruptions(df_prophet_year, n_future_years=20)
        fig_forecast_year = make_forecast_figure(df_prophet_year, prediction_year)
    else:
        fig_forecast_year = make_empty_forecast_figure("Not enough data for yearly forecast")

    return (
        fig_map,            # map-graph
        fig_impact,         # impact-graph
        fig_corr,           # corr-graph
        fig_ts,             # time-series-graph
        fig_forecast_year,  # forecast-year-graph
        total_eruptions,    # KPI
        total_deaths,       # KPI
        total_damage        # KPI
    )


if __name__ == '__main__':
    print("Launching Dashboard...")
    print("Dashboard launched at: http://127.0.0.1:7860")
    app.run(host='127.0.0.1', port=7860, debug=True)


File loaded.
Launching Dashboard...
Dashboard launched at: http://127.0.0.1:7860
